# GO-GARCH e DECO: Modelos de Fatores

**Referencias**:
- van der Weide, R. (2002). *GO-GARCH: A Multivariate Generalized Orthogonal GARCH Model*. Journal of Applied Econometrics, 17(5), 549-564.
- Engle, R. & Kelly, B. (2012). *Dynamic Equicorrelation*. Journal of Business & Economic Statistics, 30(2), 212-228.

---

Neste notebook exploramos dois modelos que abordam a **dimensionalidade** de forma diferente:

### GO-GARCH (Generalized Orthogonal GARCH)
Decompoem os retornos em **fatores ortogonais** via ICA (Independent Component Analysis).
Cada fator segue um GARCH univariado independente.

$$r_t = Z f_t, \quad H_t = Z \, \text{diag}(h_{1,t}, \ldots, h_{k,t}) \, Z'$$

### DECO (Dynamic Equicorrelation)
Simplifica o DCC assumindo uma **unica correlacao escalar** $\rho_t$ para todos os pares:

$$R_t = (1 - \rho_t) I_k + \rho_t J_k$$

onde $J_k$ e a matriz de uns ($k \times k$).

### Neste notebook

1. GO-GARCH: estimacao e fatores
2. Fatores ortogonais
3. DECO: estimacao
4. Correlacao equi-condicional dinamica
5. Comparacao de modelos

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Adicionar utils ao path
sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import (
    plot_correlation_heatmap,
)

from archbox.multivariate import CCC, DCC, DECO, GOGARCH

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Carregar dados sector_indices (5 series)
data_path = os.path.join("..", "data", "sector_indices.csv")
df = pd.read_csv(data_path, parse_dates=["date"], index_col="date")
returns = df.values
labels = [col.capitalize() for col in df.columns]
dates = df.index

print(f"Dataset: {df.shape[0]} obs x {df.shape[1]} series")
print(f"Series: {labels}")
print(f"Periodo: {df.index[0].date()} a {df.index[-1].date()}")

# Correlacao incondicional
corr = df.corr().values
plot_correlation_heatmap(corr, labels, title="Correlacao Incondicional - Setores")
plt.show()

## 1. GO-GARCH

O GO-GARCH (van der Weide, 2002) usa **Analise de Componentes Independentes (ICA)** para
decompor os retornos em fatores ortogonais:

$$r_t = Z \, f_t$$

onde:
- $Z$ e a **matriz de mistura** (mixing matrix) estimada via ICA
- $f_t$ sao os **fatores independentes** (nao apenas nao-correlacionados, mas estatisticamente independentes)

Cada fator $f_{i,t}$ segue um GARCH(1,1) univariado:
$$h_{i,t} = \omega_i + \alpha_i f_{i,t-1}^2 + \beta_i h_{i,t-1}$$

A covariancia condicional e reconstruida:
$$H_t = Z \, \text{diag}(h_{1,t}, \ldots, h_{k,t}) \, Z'$$

**Vantagem**: reduz o problema multivariado a $k$ problemas univariados independentes.

In [ ]:
# TODO: Estime GO-GARCH com archbox

# Estimar GO-GARCH com todos os componentes
model_gogarch = GOGARCH(returns, n_components=None, univariate_model="GARCH", univariate_order=(1, 1))
results_gogarch = model_gogarch.fit(method="two_step", disp=True)

print(f"\n{'='*50}")
print("GO-GARCH - Resultados")
print(f"{'='*50}")
print(f"Log-likelihood: {results_gogarch.loglike:.4f}")
print(f"AIC: {results_gogarch.aic:.4f}")
print(f"BIC: {results_gogarch.bic:.4f}")
print(f"N observacoes: {results_gogarch.n_obs}")
print(f"N series: {results_gogarch.n_series}")

## 2. Fatores ortogonais

A matriz de mistura $Z$ mapeia fatores independentes para os retornos observados.
Cada coluna de $Z$ pode ser interpretada como a **exposicao** de cada setor a um fator latente.

Os fatores capturam fontes independentes de risco:
- **Fator 1**: pode representar o risco de mercado (beta)
- **Fator 2**: pode representar um risco setorial especifico
- etc.

In [ ]:
# TODO: Extraia e visualize os fatores ortogonais

# Matriz de mistura
Z = model_gogarch._mixing_matrix
factors = model_gogarch._factors

if Z is not None:
    print("Matriz de Mistura Z (mixing matrix):")
    factor_labels = [f"Fator {i+1}" for i in range(Z.shape[1])]
    print(pd.DataFrame(Z, index=labels, columns=factor_labels).round(4))

    # Heatmap da matriz de mistura
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    im = axes[0].imshow(Z, cmap="RdBu_r", aspect="auto")
    axes[0].set_xticks(range(Z.shape[1]))
    axes[0].set_yticks(range(len(labels)))
    axes[0].set_xticklabels(factor_labels, rotation=45)
    axes[0].set_yticklabels(labels)
    axes[0].set_title("Matriz de Mistura Z")
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            axes[0].text(j, i, f"{Z[i,j]:.3f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=axes[0], shrink=0.8)

    # Fatores ao longo do tempo
    if factors is not None:
        for i in range(min(factors.shape[1], 3)):  # plotar ate 3 fatores
            axes[1].plot(dates, factors[:, i], alpha=0.6, linewidth=0.5, label=f"Fator {i+1}")
        axes[1].set_title("Fatores Ortogonais (ICA)")
        axes[1].set_xlabel("Data")
        axes[1].set_ylabel("Fator")
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("Matriz de mistura nao disponivel.")

# Volatilidades dos fatores
cond_vol = results_gogarch.conditional_volatility  # (T, k)
fig, ax = plt.subplots(figsize=(14, 6))
for i in range(cond_vol.shape[1]):
    ax.plot(dates, cond_vol[:, i], linewidth=0.8, alpha=0.8, label=labels[i])
ax.set_title("Volatilidades Condicionais (GO-GARCH)")
ax.set_xlabel("Data")
ax.set_ylabel("$\\sigma_t$")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 3. DECO - Dynamic Equicorrelation

O DECO (Engle & Kelly, 2012) simplifica drasticamente o DCC assumindo que
**todas as correlacoes sao iguais** a um escalar dinamico $\rho_t$:

$$R_t = (1 - \rho_t) I_k + \rho_t J_k$$

onde $J_k = \mathbf{1}_k \mathbf{1}_k'$ (matriz de uns).

Isso significa que a matriz de correlacao tem:
- **Diagonal**: 1 (por definicao)
- **Fora da diagonal**: $\rho_t$ (igual para todos os pares)

**Vantagem**: apenas **1 parametro** de correlacao dinamica (vs. $k(k-1)/2$ no DCC),
tornando o modelo escalavel para **centenas de ativos**.

In [ ]:
# TODO: Estime DECO com archbox

# Estimar DECO
model_deco = DECO(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_deco = model_deco.fit(method="two_step", disp=True)

print(f"\n{'='*50}")
print("DECO - Resultados")
print(f"{'='*50}")
print(f"Log-likelihood: {results_deco.loglike:.4f}")
print(f"AIC: {results_deco.aic:.4f}")
print(f"BIC: {results_deco.bic:.4f}")
print(f"Parametros: {results_deco.params}")

## 4. Correlacao equi-condicional dinamica

A correlacao escalar $\rho_t$ do DECO representa a **correlacao media** dinamica
entre todos os pares de ativos. Ela captura movimentos conjuntos do mercado:

- $\rho_t$ **alto**: todos os setores se movem juntos (crise, panico)
- $\rho_t$ **baixo**: setores se movem de forma mais independente (diversificacao efetiva)

In [ ]:
# TODO: Plote a correlacao equi-condicional ao longo do tempo

R_deco = results_deco.dynamic_correlation  # (T, k, k)

# Extrair rho_t (correlacao escalar) - pegar qualquer par off-diagonal
rho_t = R_deco[:, 0, 1]  # todas as off-diagonal sao iguais no DECO

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Correlacao equi-condicional
axes[0].plot(dates, rho_t, color="steelblue", linewidth=0.8)
axes[0].fill_between(dates, 0, rho_t, alpha=0.2, color="steelblue")
axes[0].axhline(y=np.mean(rho_t), color="red", linestyle="--", linewidth=1,
                label=f"Media: {np.mean(rho_t):.3f}")
axes[0].set_title("DECO - Correlacao Equi-Condicional Dinamica $\\rho_t$")
axes[0].set_ylabel("$\\rho_t$")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Comparar com volatilidade media do portfolio
mean_vol = np.mean(results_deco.conditional_volatility, axis=1)
axes[1].plot(dates, mean_vol, color="darkorange", linewidth=0.8)
axes[1].fill_between(dates, 0, mean_vol, alpha=0.2, color="darkorange")
axes[1].set_title("Volatilidade Media dos Setores")
axes[1].set_xlabel("Data")
axes[1].set_ylabel("$\\bar{\\sigma}_t$")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("Estatisticas de rho_t:")
print(f"  Media: {np.mean(rho_t):.4f}")
print(f"  Std: {np.std(rho_t):.4f}")
print(f"  Min: {np.min(rho_t):.4f}")
print(f"  Max: {np.max(rho_t):.4f}")

## 5. Comparacao: CCC vs DCC vs GO-GARCH vs DECO

Comparamos os quatro modelos multivariados no mesmo dataset de 5 series setoriais.

| Modelo | Correlacao | Escalabilidade | Interpretacao |
|--------|-----------|----------------|---------------|
| CCC | Constante | Alta | Simples |
| DCC | Dinamica (par a par) | Media | Rica |
| GO-GARCH | Via fatores | Alta | Fatores latentes |
| DECO | Escalar dinamica | Muito alta | Correlacao media |

In [ ]:
# TODO: Tabela comparativa com AIC, BIC e n_parametros

# Estimar CCC e DCC para comparacao
model_ccc = CCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_ccc = model_ccc.fit(method="two_step", disp=False)

model_dcc = DCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_dcc = model_dcc.fit(method="two_step", disp=False)

# Tabela comparativa
all_results = {
    "CCC": results_ccc,
    "DCC": results_dcc,
    "GO-GARCH": results_gogarch,
    "DECO": results_deco,
}

comp_table = pd.DataFrame({
    "Modelo": list(all_results.keys()),
    "Log-Likelihood": [r.loglike for r in all_results.values()],
    "AIC": [r.aic for r in all_results.values()],
    "BIC": [r.bic for r in all_results.values()],
    "N Params (corr)": [len(r.params) for r in all_results.values()],
}).set_index("Modelo")

print("="*70)
print("Comparacao de Modelos Multivariados (5 series setoriais)")
print("="*70)
print(comp_table.to_string())

# Identificar melhores modelos
best_aic = comp_table["AIC"].idxmin()
best_bic = comp_table["BIC"].idxmin()
best_ll = comp_table["Log-Likelihood"].idxmax()
print(f"\nMelhor Log-Likelihood: {best_ll}")
print(f"Melhor AIC: {best_aic}")
print(f"Melhor BIC: {best_bic}")

# Grafico de barras comparativo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = list(all_results.keys())
colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]

axes[0].bar(models, comp_table["Log-Likelihood"], color=colors)
axes[0].set_title("Log-Likelihood (maior = melhor)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(models, comp_table["AIC"], color=colors)
axes[1].set_title("AIC (menor = melhor)")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(models, comp_table["BIC"], color=colors)
axes[2].set_title("BIC (menor = melhor)")
axes[2].tick_params(axis="x", rotation=45)

fig.suptitle("Comparacao de Modelos Multivariados", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()